AI Gallery:https://pangu.huaweicloud.com/gallery/asset-detail.html?id=3d29ea9c-1a6e-44fd-85fc-3d9ff95af1c2

## MaskFormer: 推理
在这个 notebook 中，我们运行 MaskFormer

### 简要介绍： MaskFormer
MaskFormer使用掩模分类范式而不是经典的每像素分类来解决语义（和全景）分割问题。 它的网络架构与 [DETR](https://huggingface.co/docs/transformers/model_doc/detr) 类似，由三部分组成

* 骨干网络 ([Swin](https://huggingface.co/docs/transformers/model_doc/swin) or [ResNet](https://huggingface.co/docs/transformers/model_doc/resnet))

* 像素解码器，它改进了骨干网网的性能（金字塔网络结构、[FPN](https://arxiv.org/abs/1612.03144))
* a [Transformer 解码器](https://arxiv.org/abs/1706.03762).

## 使用 MaskFormerImageProcessor 预处理图像

使用两只猫在沙发上放松的图像。这是[COCO](https://cocodataset.org/#home) 目标检测验证2017数据集中的一张图像。


In [ ]:
from PIL import Image
import requests

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)
image

让我们首先使用 `MaskFormerImageProcessor` 应用常规的图像预处理。图像处理器将调整图像大小（最小尺寸=800，最大尺寸=1333），并使用ImageNet均值和标准偏差在通道间对其进行归一化。`MaskFormerImageProcessor`还将确保预处理后的图像能被某个值整除，在我们的例子中是32。这对Swin骨干网络来说是必要的。

In [ ]:
from mindnlp.transformers import MaskFormerImageProcessor

checkpoint_name = "facebook/maskformer-swin-small-coco"

processor = MaskFormerImageProcessor.from_pretrained(checkpoint_name)

In [ ]:
inputs = processor(image, return_tensors="ms")
inputs.keys()

In [ ]:
print(inputs['pixel_values'].shape)

## 前进传播
模型输入像素值和像素掩码。我们在这里使用具有Swin基础骨干网络。

In [ ]:
from mindnlp.transformers import MaskFormerForInstanceSegmentation

model = MaskFormerForInstanceSegmentation.from_pretrained(checkpoint_name)
outputs = model(**inputs)

## 后处理
从`MaskFormerForInstanceSegmentation`的输出中，我们可以创建语义和全景分割掩码。我们在`MaskFormerImageProcessor`中有不同的`post_process_*`方法。

### 语义分割

使用`processer.post_process_semantic_segmentation`创建语义分割图，其中每个像素表示类标签。

In [ ]:
semantic_segmentation = processor.post_process_semantic_segmentation(outputs)[0]
semantic_segmentation.shape

In [ ]:
semantic_segmentation

可视化

In [ ]:
%matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib import cm
import mindspore.ops as ops
import mindspore
import numpy as np
def draw_semantic_segmentation(segmentation):
    # get the used color map
    viridis = cm.get_cmap('viridis', ops.max(mindspore.tensor(segmentation.numpy()))[0])
    # get all the unique numbers
    labels_ids = ops.unique(semantic_segmentation)[0].sort()[0].tolist()
    fig, ax = plt.subplots()
    ax.imshow(segmentation.numpy())
    handles = []
    for label_id in labels_ids:
        label = model.config.id2label[label_id]
        color = viridis(label_id)
        handles.append(mpatches.Patch(color=color, label=label))
    ax.legend(handles=handles)
    return fig

draw_semantic_segmentation(semantic_segmentation)

###全景分割

你可以使用`processer.post_process_panoptic_segmentation`，它输出一个由两个键组成的字典：

- `segment`：预测的分割图，其中每个像素对应一个`segment.id`
- `segments`：一个包含以下关键字的字典列表：
    - `id`
    - `category_id`
    - `is_thing`

In [ ]:
panoptic_segmentation = processor.post_process_panoptic_segmentation(outputs)[0]
panoptic_segmentation.keys()

In [ ]:
from collections import defaultdict

def draw_panoptic_segmentation(segmentation, segments_info):
    # get the used color map
    viridis = cm.get_cmap('viridis', ops.max(mindspore.tensor(segmentation.numpy()))[0])
    fig, ax = plt.subplots()
    ax.imshow(segmentation.numpy())
    instances_counter = defaultdict(int)
    handles = []
    # for each segment, draw its legend
    for segment in segments_info:
        segment_id = segment['id']
        print(segment)
        segment_label_id = segment['label_id']
        segment_label = model.config.id2label[segment_label_id]
        label = f"{segment_label}-{instances_counter[segment_label_id]}"
        instances_counter[segment_label_id] += 1
        color = viridis(segment_id)
        handles.append(mpatches.Patch(color=color, label=label))
        
    ax.legend(handles=handles)
    return fig

draw_panoptic_segmentation(**panoptic_segmentation)